In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cuda")
print(f"device = {DEVICE}")

Newton's rate of cooling
$$\frac{dT}{dt} = -R(T - T_{env})$$

- $T$: The temperature of the coffee at time $t$ (This is what the Neural Network is trying to predict).
- $t$: Time (The input to our Neural Network).
- $T_{env}$: Ambient temperature ($25^\circ C$).
- $R$: The cooling rate constant ($0.005$).
- $\frac{dT}{dt}$: The derivative (the slope of the cooling curve).


We generate a random or uniform distribution of time points across the entire time window.

In [ ]:
# physical constants
T_env = -5.0      # Ambient temperature (-5°C)
T_initial = 100.0  # Initial temperature (100°C)
R = 0.005          # Cooling rate

In [ ]:
# generating data time (t) for 300 seconds with 150 points in column format
time_s = torch.linspace(0, 900, 900, dtype=torch.float32).view(-1, 1) # shape (150, 1)
print(time_s.shape)
print(time_s[:5])

We can get the exact solution by rearanging the equation to
$$T(t) = T_{env} + (T_{initial} - T_{env}) \cdot e^{-Rt}$$

This is required temperature equation at specified time. Solving it gives us the ground truth temperature


In [ ]:
def get_exact_solution(t):
    return T_env + (T_initial - T_env) * torch.exp(-R * t)

In [ ]:
T_data = get_exact_solution(time_s).detach()  # shape (150, 1)
print(T_data.shape)
print(T_data[:5])

Lets plot Ground Truth Temperature vs Time in second over 300 timesteps

## Plot ground truth with the PINN prediction

In [ ]:
plt.plot(time_s, T_data,
         label='Exact Physics',
         marker='o',
         markersize=1,
         markerfacecolor='red',
        #  markeredgewidth=2,
         alpha=0.5)

plt.xlabel('Time (s)')
plt.ylabel('Temperature (°C)')
plt.title('Cooling Process')
plt.legend()
plt.show()

## Build PINN Network with simple MLP

In [ ]:
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        # 1 input (t) -> 32 neurons -> 32 neurons -> 1 output (T)
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Tanh(),         # Tanh is smooth, perfect for derivatives
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, t):
        return self.net(t)

In [ ]:
# Initialize the model
model = PINN()
model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5000, gamma=0.5)

num_epochs = 200000

# Lists to store losses for plotting
training_losses = []
physics_losses = []
weight_physics_losses = []
data_losses = []

# requires autograd in time
time_s.requires_grad = True
# Move time_s and T_data to the same device as the model
time_s = time_s.to(DEVICE)
T_data = T_data.to(DEVICE)

# Training loop
for epoch in range(1, num_epochs + 1):
    start_time = time.time()

    model.train()
    optimizer.zero_grad()

    # Predict temperature from the model
    T_pred = model(time_s)

    # Compute the physics loss (residual of the ODE)
    dT_dt = torch.autograd.grad(T_pred, time_s, grad_outputs=torch.ones_like(T_pred), create_graph=True)[0]
    physics_residual = dT_dt + R * (T_pred - T_env)
    physics_loss = torch.mean(physics_residual**2)

    # Compute the data loss (MSE with observed data)
    data_loss = torch.mean((T_pred - T_data)**2)

    # Total loss
    weight_physics_loss = 1000 * physics_loss
    loss = data_loss + weight_physics_loss  # Weighted sum (give more importance to physics)

    # Backpropagation
    loss.backward()
    optimizer.step()
    scheduler.step()

    # Store losses
    training_losses.append(loss.item())
    physics_losses.append(physics_loss.item())
    weight_physics_losses.append(weight_physics_loss.item())

    data_losses.append(data_loss.item())

    if epoch == 1 or epoch % 1000 == 0 or epoch == num_epochs:
        print(f'Epoch {epoch}, Loss: {loss.item():.6f}, Time(s) = {time.time()-start_time}')
        print(f'Data Loss: {data_loss.item():.6f}, Physics Loss: {physics_loss.item():.6f}, Wt Physics Loss: {weight_physics_loss.item():.6f}')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(training_losses, label='Total Loss')
plt.plot(physics_losses, label='Physics Loss (weighted)')
plt.plot(data_losses, label='Data Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Losses Over Epochs')
plt.legend()
plt.yscale('log') # Use log scale for better visualization if losses vary greatly
plt.grid(True)
plt.show()

In [ ]:
t_test = torch.linspace(0, 2000, 2000).view(-1, 1).to(DEVICE) # Move t_test to the same device as the model
model.eval() # We use the model currently in memory
with torch.no_grad():
    T_pinn_prediction = model(t_test)
    T_exact = get_exact_solution(t_test)

plt.figure(figsize=(10, 6))
plt.plot(t_test.cpu().numpy(), T_exact.cpu().numpy(), label='Exact Physics Solution', color='black', alpha=0.3, linewidth=4) # Move back to CPU for plotting
plt.plot(t_test.cpu().numpy(), T_pinn_prediction.cpu().numpy(), label='PINN Prediction', color='blue', linestyle='--') # Move back to CPU for plotting
# plt.scatter(t_data.numpy(), T_data.numpy(), color='red', label='Observed Data (0-5 mins)', zorder=5)

plt.axvline(x=900, color='gray', linestyle=':', label='Training Data Limit')
plt.title("PINN: Coffee Cooling (Final Implementation)")
plt.xlabel("Time (seconds)")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()